In [ ]:
import os

import pandas as pd
import torch

os.chdir("..")
os.getcwd()

In [ ]:
pth = "data/s2bms/splits/s2bms_union_val_test.pth"
split_indices = torch.load(pth, weights_only=False)

In [ ]:
df_model_ready = pd.read_csv("data/s2bms/model_ready_s2bms.csv")
df_model_ready["split"] = None

df_model_ready.loc[
    df_model_ready["name_loc"].isin(list(split_indices["val_indices"])), "split"
] = "val"
df_model_ready.loc[
    df_model_ready["name_loc"].isin(list(split_indices["test_indices"])), "split"
] = "test"
df_model_ready.loc[
    df_model_ready["name_loc"].isin(list(split_indices["train_indices"])), "split"
] = "train"
df_model_ready = df_model_ready.dropna(subset=["split"])
df_model_ready["split"].value_counts(dropna=False)

keep_columns = [
    i for i in df_model_ready.columns if i in ["split", "name_loc", "lon", "lat"] or "target_" in i
]
df_model_ready[keep_columns]

os.makedirs("inference/s2bms", exist_ok=True)
df_model_ready.to_csv("inference/s2bms/targets.csv", index=False)

In [ ]:
df = pd.read_csv("data/s2bms/eo/avr_aef_256.csv")
df["split"] = None

df.loc[df["name_loc"].isin(list(split_indices["val_indices"])), "split"] = "val"
df.loc[df["name_loc"].isin(list(split_indices["test_indices"])), "split"] = "test"
df.loc[df["name_loc"].isin(list(split_indices["train_indices"])), "split"] = "train"
df = df.dropna(subset=["split"])
df["split"].value_counts(dropna=False)

In [ ]:
df = df.merge(df_model_ready[["name_loc", "lon", "lat"]], on="name_loc", how="left")
df.to_csv("inference/s2bms/avr_aef_256.csv")